> 在线讲义：[第 27 章 · 算子融合与 FlashAttention：中间结果不落地](https://wikieden.github.io/ai-infra-from-constraints/lessons/27-fusion-flash.html) · 本书《从约束推导 AI Infra》

# 第 27 章 · 算子融合与 FlashAttention：中间结果不落地

配套讲义：仓库里 `ai-infra/notes/lessons/27-fusion-flash.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 例 5-4 | 中间量 × 2 | `activation_chain_mib(fuse, fuse_quant)` | 120 / 72；156 / 108 / 60 MiB |
| 第二步 例 5-5 | 流水公式 | `pipe_time(4, 1.29, 0.28)` | 6.28 / 5.44 μs |
| 第三步 例 5-6 | 换基更新 | `online_softmax(scores, values, block)` | 7/3，与 `full_softmax` 相同 |
| 第三步 块大小 | 片上预算 | `flash_tile(b)` | (128, 64, 260, 524288) 等 |

例 5-4、双缓冲例 5-5、背压、例 5-6 在线 Softmax、注意力块大小的取舍取自 AII 第 5 章 §5.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
MiB = 2**20

**函数 `activation_chain_mib`**：G、U、T、Z 各 t MiB（BF16），量化输出 t/2。fuse_quant=None 表示没有量化这一步。

In [ ]:
def activation_chain_mib(fuse_silu_mul: bool, fuse_quant: bool | None = None, t: int = 24) -> int:
    """G、U、T、Z 各 t MiB（BF16），量化输出 t/2。fuse_quant=None 表示没有量化这一步。"""
    total = 0
    if fuse_silu_mul:
        total += 3 * t  # 读 G、U，写 Z
    else:
        total += 2 * t + 3 * t  # SiLU：读 G 写 T；乘：读 T、U 写 Z
    if fuse_quant is None:
        return total
    if fuse_quant and fuse_silu_mul:
        return 2 * t + t // 2  # 读 G、U，写量化结果
    return total + t + t // 2  # 独立量化：读 Z，写 1 字节结果

**函数 `pipe_time`**：(串行, 双缓冲) 的总时间。

In [ ]:
def pipe_time(n: int, t_m: float, t_c: float) -> tuple[float, float]:
    """(串行, 双缓冲) 的总时间。"""
    return n * (t_m + t_c), t_m + (n - 1) * max(t_m, t_c) + t_c

**函数 `online_softmax`**：逐块更新 (m, ℓ, u)，返回 (m, ℓ, u, u/ℓ)。

In [ ]:
def online_softmax(scores: list[float], values: list[float], block: int = 1) -> tuple[float, float, float, float]:
    """逐块更新 (m, ℓ, u)，返回 (m, ℓ, u, u/ℓ)。"""
    m, l, u = -math.inf, 0.0, 0.0
    for s0 in range(0, len(scores), block):
        s, v = scores[s0:s0 + block], values[s0:s0 + block]
        m_new = max(m, max(s))
        r = math.exp(m - m_new) if m != -math.inf else 0.0
        p = [math.exp(x - m_new) for x in s]
        l = r * l + sum(p)
        u = r * u + sum(pi * vi for pi, vi in zip(p, v))
        m = m_new
    return m, l, u, u / l

**函数 `full_softmax`**：小工具。

In [ ]:
def full_softmax(scores: list[float], values: list[float]) -> float:
    m = max(scores)
    w = [math.exp(x - m) for x in scores]
    return sum(wi * vi for wi, vi in zip(w, values)) / sum(w)

**函数 `flash_tile`**：K/V 块 b 行时：(可容纳的 Q 行 a, 完整扫描次数, 访问 MiB, 状态更新次数)。

In [ ]:
def flash_tile(b: int, d: int = 128, L: int = 8192, smem: int = 101_376) -> tuple[int, int, int, int]:
    """K/V 块 b 行时：(可容纳的 Q 行 a, 完整扫描次数, 访问 MiB, 状态更新次数)。"""
    # 2ad + 2bd + 4ab + 4ad + 12a ≤ smem
    a = (smem - 2 * b * d) // (2 * d + 4 * b + 4 * d + 12)
    scans = math.ceil(L / a)
    kv = 2 * L * d * 2 / MiB  # K、V 各 2 MiB
    traffic = scans * kv + 4  # 再加 Q 读、O 写
    return a, scans, int(traffic), scans * (L // b)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：例 5-4，G、U 各 24 MiB")
print(f"  分开 {activation_chain_mib(False)} MiB，融合 {activation_chain_mib(True)} MiB，时间比 {72 / 120:.1f}")
print(f"  加量化：全分开 {activation_chain_mib(False, False)}，融前两步 {activation_chain_mib(True, False)}，三步融合 {activation_chain_mib(True, True)} MiB")
print("  显存峰值：独立 SiLU 时 G、U、T 共 72 MiB；三步融合 G、U、输出 60 MiB")

In [ ]:
print("\n第二步：例 5-5，每块搬 1.29 μs、算 0.28 μs，4 块")
s, p = pipe_time(4, 1.29, 0.28)
print(f"  串行 {s:.2f} μs，双缓冲 {p:.2f} μs（−{(1 - p / s) * 100:.0f}%），计算单元忙 {4 * 0.28 / p:.0%}")
print(f"  两块同驻 RTX PRO 6000 一个 SM，每块最多 {(102_400 - 2048) // 2 // 1024} KiB")

In [ ]:
print("\n第三步：例 5-6 在线 Softmax")
m, l, u, o = online_softmax([0, math.log(2)], [1, 3])
print(f"  m = {m:.3f}，ℓ = {l}，u = {u}，输出 {o:.4f}（一次处理整行 {full_softmax([0, math.log(2)], [1, 3]):.4f} = 7/3）")
print(f"  L = 8192、d = 128：完整 S、P 各 {8192 * 8192 * 4 / MiB:.0f} MiB，写读共 {4 * 8192 * 8192 * 4 / 2**30:.0f} GiB")
for b in (1, 64, 128):
    a, scans, mib, upd = flash_tile(b)
    print(f"  b = {b:3d}：a = {a:3d}，扫描 {scans} 次，访问 {mib} MiB，更新 {upd:,} 次")

## 3. 代码里才看得到的

1. 第一块时 $m = -\infty$，$e^{-\infty - m'}$ 直接写成 0，避免 `inf - inf`。
2. 测试里用 6 个随机分数、块大小 1 / 2 / 3 / 6 都和整行 Softmax 一致：在线算法对任何分块都成立。
3. `flash_tile` 的访问量 = 扫描次数 × 4 MiB（K、V 各 2 MiB）+ 4 MiB（读 Q、写 O）。

## 4. 自己改一改

- 把 `flash_tile` 的片上预算换成 H100 的 227 KB，b = 64 时能放几行 Q？
- 给 `online_softmax` 加因果掩码：第 i 行只看前 i 个位置。
- 给 `activation_chain_mib` 加一个"融合但中间量要保留给反向"的选项，读写变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 零件直接递下一道，不回仓库省来回；读写省一半，峰值省一点；
> 洗菜慢炒菜快，两个盆够了，第三个盆白搭；
> 计票只记三个数：最高票、折算总票、折算加权和，换了最高票一起缩。